0.6529
0.6643

In [2]:
# ==============================================================
# 🧠 DistilBERT Emotion Classification — Inference (batched GPU safe)
# ==============================================================
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

# ---------------------------
# 1) 環境設定
# ---------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ---------------------------
# 2) 載入模型與 tokenizer
# ---------------------------
model_path = "best_distilbert_emotion_rdrop"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
model.eval()

# ---------------------------
# 3) 載入 LabelEncoder 類別
# ---------------------------
le = LabelEncoder()
le.classes_ = np.load(f"{model_path}/label_classes.npy", allow_pickle=True)
print("✅ Loaded LabelEncoder classes:", le.classes_)

# ---------------------------
# 4) 載入測試資料
# ---------------------------
test_df = pd.read_csv("test_ready.csv")
assert {"id", "text"}.issubset(test_df.columns), "❌ test_ready.csv 應含 id, text 欄位"
texts = list(test_df["text"].astype(str))

# ---------------------------
# 5) 分批推論設定
# ---------------------------
BATCH_SIZE = 32  # 可依 GPU VRAM 調整 (16~64 皆可)

all_preds = []

# tqdm 進度條
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Predicting"):
    batch_texts = texts[i:i + BATCH_SIZE]
    
    # Tokenize
    inputs = tokenizer(
        batch_texts,
        truncation=True,
        padding=True,
        max_length=192,
        return_tensors="pt"
    ).to(device)

    # Forward 推論
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
    
    # 手動釋放暫存 GPU 記憶體
    del inputs, outputs, logits
    torch.cuda.empty_cache()

# ---------------------------
# 6) 轉回文字標籤
# ---------------------------
pred_labels = le.inverse_transform(all_preds)

# ---------------------------
# 7) 輸出 CSV
# ---------------------------
submission = pd.DataFrame({
    "id": test_df["id"],
    "emotion": pred_labels
})

submission.to_csv("submission_distilbert_drop.csv", index=False, encoding="utf-8-sig")
print("✅ Saved: submission_distilbert_drop.csv")
print(submission.head())


Device: cuda
✅ Loaded LabelEncoder classes: ['anger' 'disgust' 'fear' 'joy' 'sadness' 'surprise']


Predicting: 100%|██████████| 509/509 [00:05<00:00, 101.72it/s]

✅ Saved: submission_distilbert_drop.csv
         id  emotion
0  0x61fc95     fear
1  0xaba820     fear
2  0x66e44d      joy
3  0xc03cf5      joy
4  0x02f65a  sadness
